In [16]:
# Step 1: Import libraries
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import pickle
from pathlib import Path

In [17]:
# Load dataset
df = pd.read_csv("../Data/raw/Students_Performance_dataset.csv")

In [18]:
# Choose text fields to represent student profiles
# We'll combine skills + interests + CGPA into a profile string
def build_profile(row):
    return f"Skills: {row['What are the skills do you have ?']}, " \
           f"Interests: {row['What is you interested area?']}, " \
           f"CGPA: {row['What is your current CGPA?']}"

df["Profile_Text"] = df.apply(build_profile, axis=1)

In [19]:
# Example scholarship descriptions (later replace with real ones)
scholarships = [
    "AI and Machine Learning Scholarship for Data Science students with CGPA above 3.5",
    "Need-based scholarship for students with family income below 30,000",
    "Networking and Cybersecurity Scholarship for students interested in security"
]


In [20]:
# Load SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [21]:
# Encode profiles and scholarships
profile_embeddings = model.encode(df["Profile_Text"].tolist(), convert_to_tensor=True)
scholarship_embeddings = model.encode(scholarships, convert_to_tensor=True)

In [22]:
# Compute similarity
similarity_scores = util.cos_sim(profile_embeddings, scholarship_embeddings)

In [23]:
# Show top matches for first student
student_idx = 0
print("Student Profile:", df.iloc[student_idx]["Profile_Text"])
for i, scholarship in enumerate(scholarships):
    print(f"Similarity with '{scholarship}': {similarity_scores[student_idx][i].item():.4f}")


Student Profile: Skills: Software Development, Interests: Data Schince, CGPA: 3.15
Similarity with 'AI and Machine Learning Scholarship for Data Science students with CGPA above 3.5': 0.5605
Similarity with 'Need-based scholarship for students with family income below 30,000': 0.2104
Similarity with 'Networking and Cybersecurity Scholarship for students interested in security': 0.3123


In [25]:

output_path = Path("../Models/sbert")
output_path.parent.mkdir(parents=True, exist_ok=True)

model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode and save
profile_embeddings = model.encode(df["Profile_Text"].tolist(), convert_to_tensor=True)
with open(output_path /"scholarship_embeddings.pkl", "wb") as f:
    pickle.dump(profile_embeddings, f)

scholarship_embeddings = model.encode(scholarships, convert_to_tensor=True)
with open(output_path/ "scholarship_embeddings.pkl", "wb") as f:
    pickle.dump(scholarship_embeddings, f)


In [26]:
from pathlib import Path
import pickle

# Define output paths
profile_path = Path(output_path /"profile_embeddings.pkl")
scholarship_path = Path(output_path /"scholarship_embeddings.pkl")
scholarship_list_path = Path(output_path/ "scholarship_list.pkl")

# Ensure directories exist
profile_path.parent.mkdir(parents=True, exist_ok=True)

# Save profile embeddings
with open(profile_path, "wb") as f:
    pickle.dump(profile_embeddings, f)
print(f"Profile embeddings saved to {profile_path}")

# Save scholarship embeddings
with open(scholarship_path, "wb") as f:
    pickle.dump(scholarship_embeddings, f)
print(f"Scholarship embeddings saved to {scholarship_path}")

# Save scholarship descriptions list (important for alignment later)
with open(scholarship_list_path, "wb") as f:
    pickle.dump(scholarships, f)
print(f"Scholarship list saved to {scholarship_list_path}")


Profile embeddings saved to ..\Models\sbert\profile_embeddings.pkl
Scholarship embeddings saved to ..\Models\sbert\scholarship_embeddings.pkl
Scholarship list saved to ..\Models\sbert\scholarship_list.pkl
